<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/JS11/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load Dataset
df = pd.read_csv('/content/voice.csv')

# 2. Separasi Fitur (X) dan Target (y)
X = df.drop('label', axis=1)
y = df['label']

# Encode target ('male'/'female') menjadi numerik (1/0)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# 3. Definisikan Kernel dan Rasio Split
kernels = ['linear', 'poly', 'rbf']
split_ratios = [0.3, 0.2]  # 0.3 = 70:30, 0.2 = 80:20
results = []

# 4. Iterasi Pelatihan Model & Evaluasi
for test_size in split_ratios:
    train_ratio = int((1 - test_size) * 100)
    test_ratio = int(test_size * 100)
    split_label = f"{train_ratio}:{test_ratio}"

    # Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # Feature Scaling (Penting untuk SVM)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for kernel in kernels:
        # Inisialisasi & Fit Model SVM
        clf = SVC(kernel=kernel, random_state=42)
        clf.fit(X_train_scaled, y_train)

        # Prediksi & Hitung Akurasi
        y_pred = clf.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': split_label,
            'Kernel': kernel.upper(),
            'Akurasi (%)': round(acc * 100, 2)
        })

# 5. Tampilkan Hasil Tabulasi
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Rasio Split Kernel  Akurasi (%)
      70:30 LINEAR        97.90
      70:30   POLY        96.00
      70:30    RBF        98.32
      80:20 LINEAR        97.48
      80:20   POLY        95.58
      80:20    RBF        98.26


In [13]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Fungsi Ekstraksi Fitur Histogram Warna
def extract_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)
    if image is None:
        return None
    # Konversi BGR ke HSV untuk representasi pencahayaan yang lebih stabil
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # Hitung 3D Color Histogram
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    # Normalisasi histogram
    cv2.normalize(hist, hist)
    return hist.flatten()

# 2. Load Dataset Citra & Ekstrasi Fitur
# (Kustomisasi 'dataset_dir' sesuai struktur folder Anda: ./day_night/day dan ./day_night/night)
def load_image_dataset(dataset_dir):
    data = []
    labels = []

    categories = {'day': 0, 'night': 1}
    for category, label in categories.items():
        folder_path = os.path.join(dataset_dir, category)
        if not os.path.exists(folder_path):
            continue
        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            hist = extract_histogram(img_path)
            if hist is not None:
                data.append(hist)
                labels.append(label)

    return np.array(data), np.array(labels)


# 1. Arahkan path ke folder yang langsung berisi subfolder 'day' dan 'night'
dataset_path = 'data_image_day_night/images/test'  # Atau 'data_image_day_night/images/train'

# 2. Load dataset
X_img, y_img = load_image_dataset(dataset_path)

# 3. Jalankan pembagian dataset
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42, stratify=y_img
)

# 4. Feature Scaling
scaler_img = StandardScaler()
X_train_img_scaled = scaler_img.fit_transform(X_train_img)
X_test_img_scaled = scaler_img.transform(X_test_img)

# 5. Hyperparameter Tuning (GridSearchCV) untuk Kernel RBF
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_img_scaled, y_train_img)

# 6. Evaluasi Model Terbaik
best_svm = grid_search.best_estimator_
y_pred_img = best_svm.predict(X_test_img_scaled)
best_accuracy = accuracy_score(y_test_img, y_pred_img)

print("=== HASIL EXPERIMEN CLASSIFICATION SIANG / MALAM ===")
print(f"Hyperparameter Terbaik : {grid_search.best_params_}")
print(f"Akurasi Data Uji (80:20): {best_accuracy * 100:.2f}%")
print("\nLaporan Klasifikasi Detail:")
print(classification_report(y_test_img, y_pred_img, target_names=['Siang (Day)', 'Malam (Night)']))

=== HASIL EXPERIMEN CLASSIFICATION SIANG / MALAM ===
Hyperparameter Terbaik : {'C': 10, 'gamma': 0.001}
Akurasi Data Uji (80:20): 87.50%

Laporan Klasifikasi Detail:
               precision    recall  f1-score   support

  Siang (Day)       1.00      0.75      0.86        16
Malam (Night)       0.80      1.00      0.89        16

     accuracy                           0.88        32
    macro avg       0.90      0.88      0.87        32
 weighted avg       0.90      0.88      0.87        32

